In [ ]:
import numpy as np
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import time
from numba import cuda

BLOCK_SIZE = 16   # 2D block for map kernels
RBLOCK = 256      # 1D block for reduction kernel

img = mpimg.imread('original_image.jpg')
height, width, channels = img.shape
print("Image shape:", img.shape)

# Labwork 7: Grayscale Stretch
1. Convert to gray (MAP)
2. Find min/max intensity (REDUCE)
3. Stretch each pixel to [0, 255] (MAP)

In [ ]:
# Step 1: MAP - RGB to gray
@cuda.jit
def gray_kernel(src, dst):
    x, y = cuda.grid(2)
    h, w, _ = src.shape
    if x < w and y < h:
        dst[y, x] = (float(src[y, x, 0]) + src[y, x, 1] + src[y, x, 2]) / 3


# Step 2: REDUCE - each block finds the min and max of its 256 pixels
@cuda.jit
def minmax_kernel(src, dst_min, dst_max):
    cmin = cuda.shared.array(RBLOCK, np.float32)
    cmax = cuda.shared.array(RBLOCK, np.float32)
    t = cuda.threadIdx.x
    i = cuda.grid(1)
    if i >= src.size:
        i = 0  # out of range: reuse pixel 0, it does not change min/max
    cmin[t] = src[i]
    cmax[t] = src[i]
    cuda.syncthreads()

    s = cuda.blockDim.x // 2
    while s > 0:
        if t < s:
            cmin[t] = min(cmin[t], cmin[t + s])
            cmax[t] = max(cmax[t], cmax[t + s])
        cuda.syncthreads()
        s //= 2

    if t == 0:
        dst_min[cuda.blockIdx.x] = cmin[0]
        dst_max[cuda.blockIdx.x] = cmax[0]


# Step 3: MAP - linear stretch
@cuda.jit
def stretch_kernel(src, dst, gmin, gmax):
    x, y = cuda.grid(2)
    h, w = src.shape
    if x < w and y < h:
        dst[y, x] = (src[y, x] - gmin) / (gmax - gmin) * 255


def gpu_minmax(d_gray):
    n = d_gray.size
    blocks = (n + RBLOCK - 1) // RBLOCK
    d_min = cuda.device_array(blocks, np.float32)
    d_max = cuda.device_array(blocks, np.float32)
    minmax_kernel[blocks, RBLOCK](d_gray.reshape(n), d_min, d_max)
    # few block results left: finish on CPU
    return d_min.copy_to_host().min(), d_max.copy_to_host().max()


def grayscale_stretch(img, bs):
    block = (bs, bs)
    grid = ((width + bs - 1) // bs, (height + bs - 1) // bs)
    d_src = cuda.to_device(img)
    d_gray = cuda.device_array((height, width), np.float32)
    d_out = cuda.device_array((height, width), np.uint8)

    gray_kernel[grid, block](d_src, d_gray)
    gmin, gmax = gpu_minmax(d_gray)
    stretch_kernel[grid, block](d_gray, d_out, gmin, max(gmax, gmin + 1))
    cuda.synchronize()
    return d_gray.copy_to_host(), d_out.copy_to_host(), gmin, gmax

In [ ]:
grayscale_stretch(img, BLOCK_SIZE)

start = time.time()
gray, stretched, gmin, gmax = grayscale_stretch(img, BLOCK_SIZE)
print(f"GPU time: {time.time() - start:.4f}s")
print(f"GPU min/max: {gmin:.2f} / {gmax:.2f}")
print(f"NumPy check: {gray.min():.2f} / {gray.max():.2f}")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(gray, cmap='gray', vmin=0, vmax=255)
plt.title(f"Gray [{gmin:.0f}, {gmax:.0f}]")
plt.subplot(1, 2, 2)
plt.imshow(stretched, cmap='gray', vmin=0, vmax=255)
plt.title("Stretched [0, 255]")
plt.savefig('stretch_output.png', bbox_inches='tight')
plt.show()